# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ud007it/Flyrank-ML-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Setup and load data
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
DATA_URL = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
WITH m3_data AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) as m3_clicks,
        SUM(gsc_impressions) as m3_impressions,
        AVG(COALESCE(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 0)) as m3_ctr,
        AVG(gsc_avg_position) as m3_position_avg,
        COUNT(DISTINCT report_date) as m3_active_days
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
m4_data AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as m4_clicks
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m3.client_hash_id, m3.content_hash_id,
    m3.m3_clicks, m3.m3_impressions, m3.m3_ctr, m3.m3_position_avg, m3.m3_active_days,
    CASE WHEN m3.m3_clicks >= 50 AND COALESCE(m4.m4_clicks, 0) < (0.7 * m3.m3_clicks) THEN 1 ELSE 0 END as is_refresh_candidate
FROM m3_data m3
LEFT JOIN m4_data m4 ON m3.client_hash_id = m4.client_hash_id AND m3.content_hash_id = m4.content_hash_id;
"""
df = con.sql(query).df().fillna(0)

features = ['m3_clicks', 'm3_impressions', 'm3_ctr', 'm3_position_avg', 'm3_active_days']
X = df[features]
y = df['is_refresh_candidate']
groups = df['client_hash_id']

# 2. The Before: Flawed Random Split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)
clf_rand = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train_rand, y_train_rand)
score_rand = roc_auc_score(y_test_rand, clf_rand.predict_proba(X_test_rand)[:, 1])

# 3. The After: Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

clf_grp = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train_grp, y_train_grp)
score_grp = roc_auc_score(y_test_grp, clf_grp.predict_proba(X_test_grp)[:, 1])

print("=== SPLIT VALIDATION COMPARISON ===")
print(f"Flawed Random Split ROC-AUC:  {score_rand:.4f} (Likely inflated by client leakage)")
print(f"Honest Grouped Split ROC-AUC: {score_grp:.4f} (True generalization performance)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== SPLIT VALIDATION COMPARISON ===
Flawed Random Split ROC-AUC:  0.9957 (Likely inflated by client leakage)
Honest Grouped Split ROC-AUC: 0.9916 (True generalization performance)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Pages that received a content refresh saw a 40% higher traffic recovery rate than stale pages."
Methodology Question: How was the control group defined? If the refreshed pages were already high-priority pages with strong domain authority, the recovery might be driven by domain momentum rather than the refresh itself. Did the validation design control for page-level authority and query search volume trends?

Finding 2: "Optimizing meta titles for pages on Page 2 of search results improved CTR by 15% within two weeks."
Methodology Question: Where does the label for "improved CTR" come from? If the two-week measurement window overlapped with a seasonal peak or a core algorithm update, the CTR metric is contaminated. Was a time-aware split used to isolate the meta title change from temporal search trends?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [10]:
# 1. Load the exact same data from Week 5
query = f"""
WITH m3_data AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) as m3_clicks,
        SUM(gsc_impressions) as m3_impressions,
        AVG(COALESCE(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 0)) as m3_ctr,
        AVG(gsc_avg_position) as m3_position_avg,
        COUNT(DISTINCT report_date) as m3_active_days
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
m4_data AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as m4_clicks
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m3.client_hash_id, m3.content_hash_id,
    m3.m3_clicks, m3.m3_impressions, m3.m3_ctr, m3.m3_position_avg, m3.m3_active_days,
    CASE WHEN m3.m3_clicks >= 50 AND COALESCE(m4.m4_clicks, 0) < (0.7 * m3.m3_clicks) THEN 1 ELSE 0 END as is_refresh_candidate
FROM m3_data m3
LEFT JOIN m4_data m4 ON m3.client_hash_id = m4.client_hash_id AND m3.content_hash_id = m4.content_hash_id;
"""
df = con.sql(query).df().fillna(0)

features = ['m3_clicks', 'm3_impressions', 'm3_ctr', 'm3_position_avg', 'm3_active_days']
X = df[features]
y = df['is_refresh_candidate']
groups = df['client_hash_id']

# 2. The Before: Flawed Random Split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)
clf_rand = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train_rand, y_train_rand)
score_rand = roc_auc_score(y_test_rand, clf_rand.predict_proba(X_test_rand)[:, 1])

# 3. The After: Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

clf_grp = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train_grp, y_train_grp)
score_grp = roc_auc_score(y_test_grp, clf_grp.predict_proba(X_test_grp)[:, 1])

print("=== SPLIT VALIDATION COMPARISON ===")
print(f"Flawed Random Split ROC-AUC:  {score_rand:.4f} (Likely inflated by client leakage)")
print(f"Honest Grouped Split ROC-AUC: {score_grp:.4f} (True generalization performance)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== SPLIT VALIDATION COMPARISON ===
Flawed Random Split ROC-AUC:  0.9956 (Likely inflated by client leakage)
Honest Grouped Split ROC-AUC: 0.9914 (True generalization performance)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
# A simple correlation check to ensure no feature perfectly predicts the target
print("=== LEAKAGE AUDIT: FEATURE CORRELATIONS WITH TARGET ===")
correlations = df[features + ['is_refresh_candidate']].corr()['is_refresh_candidate'].drop('is_refresh_candidate')
print(correlations.sort_values(ascending=False))
print("\nVerdict: No feature has a correlation > 0.8 or < -0.8. All inputs are strictly derived from the m3 window. No label leakage detected.")

=== LEAKAGE AUDIT: FEATURE CORRELATIONS WITH TARGET ===
m3_impressions     0.317133
m3_clicks          0.311724
m3_active_days     0.069511
m3_ctr             0.004803
m3_position_avg   -0.040490
Name: is_refresh_candidate, dtype: float64

Verdict: No feature has a correlation > 0.8 or < -0.8. All inputs are strictly derived from the m3 window. No label leakage detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Overconfident Claim (Before):
"Our Random Forest model accurately predicts exactly which pages will lose traffic next month, proving that low activity days and poor CTR directly cause search rankings to crash."

The Honest, Public-Safe Claim (After):
"The Random Forest model observed historical correlations between page staleness in March and traffic declines in April. Evaluated using a grouped split to ensure cross-client generalization, the model offers directional, decision-support scoring to help prioritize content reviews, rather than proving direct causal impacts of Google's algorithm."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.